# Recruitment Assistant Chatbot
### Advanced Prompt Engineering with LangChain and Google Gemini

**Author:** _[Jeffrey Saad]_ &nbsp;|&nbsp; **Assignment:** Assignment 3 – Advanced Prompt Engineering

---

## Overview

This notebook implements a conversational assistant for a recruitment agency. The assistant answers career-related questions in a friendly, concise tone and extracts structured candidate information (name, desired role and years of experience) as JSON for integration with HR systems.

## Requirements Coverage

| Requirement | Implementation | Section |
|---|---|---|
| Role prompting | System prompt defining the assistant's role, tone and rules | 4 |
| Few-shot prompting | Four labelled examples embedded in the system prompt | 4 |
| Structured JSON output | Pydantic `CandidateInfo` schema serialised with `json.dumps` | 5, 8 |
| Function calling | `extract_candidate_info` tool bound to the model with `bind_tools` | 6 |
| Conversational memory | Per-session message buffer | 7 |
| Testing | Safeguard checks plus 12 live scenarios: career questions, extraction, memory, corrections, invented-detail checks and session isolation | 9 |
| User interface (optional) | Gradio chat interface | 10 |
| Reflection | 100–300 word reflection | 11 |

## Technology Stack

| Component | Technology |
|---|---|
| Language model | Google Gemini (`gemini-3.5-flash`; backup `gemini-flash-lite-latest`) |
| Framework | LangChain (`langchain-core`, `langchain-google-genai`) |
| Validation | Pydantic v2 |
| Interface | Gradio (optional) |

## Architecture

```
User message
     │
     ▼
Prompt: system prompt (role + rules + few-shot examples)
        + conversation history (memory buffer)
        + current message
     │
     ▼
Gemini with the extract_candidate_info tool bound
     │
     ├── no tool call ──► conversational text ─────────────────────────────┐
     │                                                                      │
     └── tool call ─────► grounding check ──► Pydantic validation ──► JSON ──┤
                                                                            ▼
                                                     memory buffer and profile updated
```

## Running the Notebook

1. Obtain a Gemini API key from [Google AI Studio](https://aistudio.google.com/apikey).
2. Store the key as a Colab secret named `GOOGLE_API_KEY`, or enter it when prompted in Section 3.
3. Select **Runtime → Run all**.

## 1. Install Libraries

The core chatbot depends on two packages. Pydantic is installed as a LangChain dependency, and Gradio is installed separately in Section 10 so that the core implementation does not depend on it.

| Package | Purpose |
|---|---|
| `langchain-core` | Prompt templates, message types, tools and runnables |
| `langchain-google-genai` | LangChain integration for Gemini chat models |

In [ ]:
!pip install -qU langchain-core langchain-google-genai

## 2. Imports

Standard library, Pydantic and LangChain imports used throughout the notebook.

In [ ]:
# Standard library
import json
import os
import re
from getpass import getpass
from typing import Optional

# Data validation (used for the candidate schema)
from pydantic import BaseModel, ConfigDict, Field, ValidationError, field_validator

# LangChain building blocks
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

# Gemini chat model integration
from langchain_google_genai import ChatGoogleGenerativeAI

## 3. API Key Setup and LLM Connection

**API key.** The key is not stored in the notebook. It is read from Colab Secrets (`GOOGLE_API_KEY`) when available and otherwise requested through `getpass`, which hides the input.

**Model configuration.**

| Setting | Value | Notes |
|---|---|---|
| Primary model | `gemini-3.5-flash` | Fast, low-cost and available on the free tier |
| Backup model | `gemini-flash-lite-latest` | Used automatically through `with_fallbacks` if the primary model fails |
| `max_retries` | 6 | Total attempts per request; `503 UNAVAILABLE` and `429` responses are retried with exponential backoff |

The second cell performs a connection check.

In [ ]:
# --- API key (kept out of the code) ---
if not os.environ.get("GOOGLE_API_KEY"):
    try:
        from google.colab import userdata  # Colab "Secrets" panel
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except Exception:
        os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

In [ ]:
# --- Connect to the LLM ---
MODEL_NAME = "gemini-3.5-flash"                # main model (alternatives: "gemini-flash-latest", "gemini-2.5-flash")
BACKUP_MODEL_NAME = "gemini-flash-lite-latest"  # used only if the main model is overloaded / unavailable

# max_retries = total attempts per request; 503 and 429 errors are retried with backoff
llm = ChatGoogleGenerativeAI(model=MODEL_NAME, max_retries=6)
backup_llm = ChatGoogleGenerativeAI(model=BACKUP_MODEL_NAME, max_retries=3)

# Quick connection check (falls back to the backup model if the main one is busy)
print(llm.with_fallbacks([backup_llm]).invoke("Reply with exactly: Connected to Gemini!").text)

## 4. Recruitment Assistant Prompt

The system prompt combines three prompt-engineering techniques:

- **Role prompting:** defines the assistant as *"a friendly recruitment assistant helping job seekers with career questions and extracting their basic information for job applications."*
- **Behavioural rules:** separate the two kinds of message the assistant handles.
  - **Career questions** are always answered in friendly, concise plain text (a one-sentence opening followed by 3–5 bullet points). This applies even when a candidate profile already exists, and the assistant does not ask for personal details first.
  - **Candidate details** trigger the tool only when the current message states or corrects the user's own name, desired role or years of experience. Only details the user has actually given are used. Experience is recorded only when a duration is stated, and an explicit "no experience" is recorded as `0`.
- **Few-shot examples:** six examples covering career advice, a single-message extraction, a two-turn extraction, a career question asked after a profile exists, and a question about experience that must not produce an invented value.

**Design decisions**

- The few-shot examples are embedded in the system prompt and explicitly marked as fictional, rather than supplied as `HumanMessage`/`AIMessage` pairs. This keeps example candidates out of the live conversation.
- The example names (Maya, Priya) do not appear in any test. An earlier version reused "Sam" from the examples, and the model copied that name into profiles for users who had never given one.

The prompt template is ordered as: **system prompt → conversation history → current user message**.

In [ ]:
SYSTEM_PROMPT = """You are a friendly recruitment assistant helping job seekers with career questions and extracting their basic information for job applications. You work for a recruitment agency.

## Two kinds of messages
1. Career questions (interviews, CVs, skills, job search, career changes, what experience a job needs, and so on).
   - Answer them in friendly, concise plain text, even if you already know the candidate's details.
   - Do not call the tool for a career question.
   - Do not ask for the user's name, role or experience before or after answering. Do not repeat their profile.
2. Candidate details: the user's CURRENT message states or corrects their own name, the role they are applying for, or their years of experience.
   - Call the extract_candidate_info tool.

## Tone and format for career questions
- Be friendly, encouraging and professional, but concise (no more than about 120 words).
- Give practical, actionable advice: one short opening sentence, then 3-5 bullet points.
- Stay focused on careers. If a question is unrelated, politely steer the conversation back to careers.
- Never answer a career question with JSON.

## Rules for extract_candidate_info
- Use only details the user has stated about themselves in this conversation. Never invent, guess or assume anything.
- The people in the examples below are fictional. Never use their names or details.
- Pass the complete, up-to-date details each time: combine details from earlier messages with the new ones. When the user corrects a detail, use the new value and keep the other details unchanged.
- If you know the user's name or role but not both, do not call the tool. Reply briefly and ask only for the missing detail.
- experience_years: include it only if the user stated a duration (for example "three years" -> 3). If they say they have no experience, use 0. If no duration was stated, leave the field out.
- desired_role: a clean job title in Title Case, without words like "role" or "position" (for example "a developer role" -> "Developer").
- Always use the real tool call. Never write the tool call or JSON as plain text.

## Examples (fictional people, shown for style only; they are NOT part of the real conversation)

Example 1 (career question)
User: How do I prepare for a data analyst interview?
Assistant: Great choice of role! Focus on these areas:
- SQL: practise joins, GROUP BY and window functions.
- Excel or Python: be ready to clean and summarise a small dataset.
- Statistics: review averages, distributions and A/B testing basics.
- Storytelling: prepare 2-3 projects you can explain with the STAR method.
Good luck, you've got this!

Example 2 (career question)
User: What skills should I learn to become a software developer?
Assistant: Here's a practical starting path:
- Learn one language well (Python or JavaScript are great first picks).
- Use Git and GitHub for all your projects.
- Practise data structures and problem solving.
- Build 2-3 small projects you can show employers.
Consistency beats cramming, so a little every day goes a long way!

Example 3 (candidate details in one message)
User: My name is Maya, I'm applying for a nurse role and I have five years of experience.
Assistant action: call extract_candidate_info with name="Maya", desired_role="Nurse", experience_years=5

Example 4 (details across two messages, no duration stated)
User: My name is Priya.
Assistant: Nice to meet you, Priya! What role are you interested in?
User: I'd like a UX designer position.
Assistant action: call extract_candidate_info with name="Priya", desired_role="UX Designer" (experience_years is left out because no duration was stated)

Example 5 (career question after the profile was recorded)
User: How can I improve my design portfolio?
Assistant: A strong portfolio shows how you think, not just the final screens:
- Pick 3-4 of your best projects rather than everything.
- For each one, explain the problem, your process and the result.
- Include sketches or wireframes to show your thinking.
- Keep the layout clean and easy to scan.
(No tool call and no request for personal details.)

Example 6 (question about experience, no duration stated)
User: What experience is needed for a junior UX role?
Assistant: Junior UX roles focus more on potential than on years:
- A portfolio with 2-3 case studies, even from courses or personal projects.
- Familiarity with Figma and basic user research methods.
- Internships, freelance work or volunteer projects all count.
(No tool call. Never turn a question like this into an experience_years value.)
"""

# system prompt -> memory buffer -> new user message
# (SystemMessage is used directly so the prompt text is not treated as a template,
#  which means literal braces in it would never break formatting.)
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(content=SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

print(f"System prompt length: {len(SYSTEM_PROMPT.split())} words")
print("Prompt input variables:", prompt.input_variables)

## 5. Candidate Information Schema

The function schema specified in the assignment is reproduced below for reference and implemented as the Pydantic model `CandidateInfo`. LangChain derives the tool schema from this model, and Pydantic validates every extraction.

| Field | Type | Required | Validation |
|---|---|---|---|
| `name` | string | Yes | Whitespace stripped before validation; blank values rejected |
| `desired_role` | string | Yes | Whitespace stripped; blank values rejected; normalised to Title Case (acronyms such as "UX" preserved) |
| `experience_years` | integer | No | Range 0–60; defaults to `None`; an explicit `0` is kept |

`str_strip_whitespace=True` strips string values *before* the `min_length` check runs, so a name such as `"   "` is rejected rather than accepted as an empty string. Records are serialised with `exclude_none=True`, so `experience_years` is omitted from the JSON output when the candidate has not stated it.

In [ ]:
# The schema from the assignment brief (kept for reference / comparison)
EXTRACT_CANDIDATE_INFO_SCHEMA = {
    "name": "extract_candidate_info",
    "description": "Extract candidate information for job applications",
    "parameters": {
        "type": "object",
        "properties": {
            "name": {"type": "string"},
            "desired_role": {"type": "string"},
            "experience_years": {"type": "integer"},
        },
        "required": ["name", "desired_role"],
    },
}


class CandidateInfo(BaseModel):
    """Extract candidate information for job applications."""

    # Strip whitespace before the min_length check, so blank strings are rejected
    model_config = ConfigDict(str_strip_whitespace=True)

    name: str = Field(
        min_length=1,
        description="The candidate's name exactly as the user stated it. Never use a name the user did not give.",
    )
    desired_role: str = Field(
        min_length=1,
        description="The job title the user said they are applying for, in Title Case, e.g. 'Data Analyst'.",
    )
    experience_years: Optional[int] = Field(
        default=None,
        ge=0,
        le=60,
        description=(
            "Years of professional experience as a whole number, only if the user stated a duration "
            "(use 0 if they said they have no experience). Leave it out otherwise. Never guess."
        ),
    )

    @field_validator("desired_role")
    @classmethod
    def title_case_role(cls, value: str) -> str:
        # Capitalise the first letter of each word but keep acronyms like "UX" or "QA" intact
        return " ".join(word[:1].upper() + word[1:] for word in value.split())


# Quick validation checks (no LLM needed)
print(CandidateInfo(name=" Sam ", desired_role="developer", experience_years=3).model_dump(exclude_none=True))
print(CandidateInfo(name="Sarah", desired_role="marketing analyst").model_dump(exclude_none=True))
print(CandidateInfo(name="Lee", desired_role="junior tester", experience_years=0).model_dump(exclude_none=True))
try:
    CandidateInfo(name="   ", desired_role="developer")
except ValidationError as err:
    print("Blank name rejected:", err.errors()[0]["type"])

## 6. Function Calling

The `@tool` decorator registers `extract_candidate_info` as a LangChain tool with `CandidateInfo` as its argument schema. `bind_tools` attaches the tool definition to every request sent to the primary and backup models.

**Execution flow**

1. The model decides whether the current message supplies or corrects candidate details, or is a career question.
2. For candidate details, the model returns a tool call (`response.tool_calls`) containing schema-conformant arguments instead of text.
3. The application checks the arguments against what the user actually said (Section 8), merges them with the saved profile, and executes the tool. The tool validates the arguments and returns a dictionary for JSON serialisation.

The cell below verifies that the generated tool schema matches the assignment schema, invokes the tool directly, and displays a raw tool call returned by the model.

In [ ]:
@tool("extract_candidate_info", args_schema=CandidateInfo)
def extract_candidate_info(name: str, desired_role: str, experience_years: Optional[int] = None) -> dict:
    """Extract candidate information for job applications."""
    candidate = CandidateInfo(name=name, desired_role=desired_role, experience_years=experience_years)
    # exclude_none=True -> experience_years is omitted (not 0) when it wasn't provided
    return candidate.model_dump(exclude_none=True)


# Give the model access to the tool (the backup model gets the same tool, in case Gemini is overloaded)
llm_with_tools = llm.bind_tools([extract_candidate_info]).with_fallbacks(
    [backup_llm.bind_tools([extract_candidate_info])]
)

# --- Check: generated schema vs. the assignment schema ---
generated = convert_to_openai_tool(extract_candidate_info)["function"]
print("Tool name:        ", generated["name"])
print("Description:      ", generated["description"])
print("Properties match: ", set(generated["parameters"]["properties"]) == set(EXTRACT_CANDIDATE_INFO_SCHEMA["parameters"]["properties"]))
print("Required match:   ", generated["parameters"]["required"] == EXTRACT_CANDIDATE_INFO_SCHEMA["parameters"]["required"])

# --- Check: run the tool directly (no LLM) ---
print("\nDirect tool call:", extract_candidate_info.invoke({"name": "Sam", "desired_role": "developer", "experience_years": 3}))

# --- Check: let the model decide to call the tool ---
raw = llm_with_tools.invoke("My name is Sam and I am applying for a software developer role with 3 years of experience.")
print("\nRaw tool calls returned by the model:")
print(raw.tool_calls)

## 7. Conversational Memory

The assignment specifies a memory buffer. `ConversationBufferMemory` has been removed in LangChain 1.x, and its successors, `RunnableWithMessageHistory` and `InMemoryChatMessageHistory`, are deprecated in favour of LangGraph persistence. A graph runtime is unnecessary for a single-model chatbot, so the buffer is implemented directly with equivalent behaviour:

- Each `session_id` maps to an ordered list of LangChain messages (`HumanMessage`, `AIMessage`).
- Every exchange is appended to the buffer.
- The complete buffer is injected into the prompt through `MessagesPlaceholder("history")` on each request.

Alongside the message buffer, each session keeps its **candidate profile**: the validated details recorded so far. A correction updates only the fields that changed, and the other details are retained.

Separate session IDs isolate conversations, so details from one candidate cannot appear in another candidate's record. `clear_memory` removes both the messages and the profile.

In [ ]:
# session_id -> list of messages (the conversation buffer)
memory_store: dict[str, list[BaseMessage]] = {}

# session_id -> candidate details recorded so far (may be partial until name and role are known)
profile_store: dict[str, dict] = {}


def get_memory(session_id: str) -> list[BaseMessage]:
    """Return the message buffer for a session, creating it if needed."""
    return memory_store.setdefault(session_id, [])


def get_profile(session_id: str) -> dict:
    """Return the candidate details recorded for a session (empty if none)."""
    return profile_store.setdefault(session_id, {})


def save_to_memory(session_id: str, user_message: str, assistant_message: str) -> None:
    """Append one user/assistant exchange to the buffer."""
    get_memory(session_id).extend([
        HumanMessage(content=user_message),
        AIMessage(content=assistant_message),
    ])


def clear_memory(session_id: str) -> None:
    """Forget a conversation and its candidate profile."""
    memory_store.pop(session_id, None)
    profile_store.pop(session_id, None)


def show_memory(session_id: str) -> None:
    """Print the buffer and profile, handy for checking what the bot remembers."""
    print(f"--- Memory for session '{session_id}' ---")
    for msg in get_memory(session_id):
        speaker = "User" if isinstance(msg, HumanMessage) else "Assistant"
        print(f"{speaker}: {msg.content}")
    print(f"Profile: {get_profile(session_id)}")


# Tiny demo of the buffer (no LLM call)
save_to_memory("demo", "Hi there!", "Hello! How can I help with your career today?")
show_memory("demo")
clear_memory("demo")

## 8. Chatbot

`chatbot_response(message, session_id)` implements the complete request cycle:

1. Retrieve the conversation history for the session.
2. Invoke the chain `prompt | llm_with_tools` with the history and the new message.
3. If the model returns an `extract_candidate_info` call, apply the safeguards below, merge the result with the saved profile, and validate it with the tool.
4. Return formatted JSON (`json.dumps(..., indent=2)`) when the profile is created or updated. Otherwise return plain text.
5. Append the exchange to the memory buffer.

**Safeguards against invented or misplaced extractions**

The prompt tells the model what to do. The code checks that the result is justified by what the user actually wrote:

| Safeguard | Rule |
|---|---|
| Grounded name | Every word of `name` must appear in the user's own messages in this session. Names from few-shot examples or other sessions are discarded. |
| Grounded role | Every word of `desired_role` must match a word the user wrote (allowing endings such as "engineering" → "Engineer"). |
| Grounded experience | `experience_years` is kept only if the user stated that duration (e.g. "3 years", "three years"). An explicit "no experience" or "0 years" gives `0`. Mentioning "experience" or "years" without a number is not enough. |
| Career questions | A tool call on a question that contains no self-description (such as "my name is", "I am", "I have") is ignored, and the question is answered in plain text. |
| No new information | A tool call that would not change the saved profile is ignored, so repeated profile JSON is never returned for ordinary messages. |

**Validation and error handling**

| Scenario | Handling |
|---|---|
| Missing, blank or invalid required field | The valid details are kept and the assistant asks only for the missing or invalid ones. For a question, it answers the question instead. |
| JSON written as text instead of a tool call | Parsed and passed through the same safeguards |
| API unavailable | Retried, then routed to the backup model. If both fail, an error message is returned and memory is left unchanged. |

For extraction turns, memory stores a short plain-text summary of the recorded details rather than raw JSON. This keeps later conversational replies free of JSON formatting.

In [ ]:
# The chain: prompt template -> Gemini (with the tool bound)
chatbot_chain = prompt | llm_with_tools
# Same prompt without the tool: used when a tool call is rejected and a plain answer is needed
answer_chain = prompt | llm.with_fallbacks([backup_llm])

ERROR_PREFIX = "Sorry, I couldn't reach the language model"
FIELD_LABELS = {
    "name": "your name",
    "desired_role": "the role you're applying for",
    "experience_years": "your years of experience as a whole number",
}

# --- Grounding helpers: was a value actually stated by the user? ---
NUMBER_WORDS = {
    "zero": 0, "one": 1, "a": 1, "an": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
    "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13,
    "fourteen": 14, "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18,
    "nineteen": 19, "twenty": 20,
}
DURATION_PATTERN = re.compile(r"\b(\d{1,2}|" + "|".join(NUMBER_WORDS) + r")\s*\+?\s*(?:full[- ]time\s+)?(?:years?|yrs?)\b")
NO_EXPERIENCE_PATTERN = re.compile(r"\b(?:no|zero)\s+(?:prior\s+|previous\s+|professional\s+|relevant\s+)?(?:work\s+)?experience\b")
SELF_DESCRIPTION_PATTERN = re.compile(r"\b(?:my name|i am|i'm|im|i have|i've|call me|i'd like|i would like|i want to apply|applying)\b")


def words(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", str(text).lower())


def stated_years(user_text: str) -> set[int]:
    """All experience durations the user explicitly stated, e.g. 'three years' -> {3}."""
    text = user_text.lower()
    values = {int(v) if v.isdigit() else NUMBER_WORDS[v] for v in DURATION_PATTERN.findall(text)}
    if NO_EXPERIENCE_PATTERN.search(text):
        values.add(0)
    return values


def is_stated(value, user_text: str, allow_word_endings: bool = False) -> bool:
    """True if every word of value appears in the user's text (optionally allowing different endings)."""
    value_words, user_words = words(value), set(words(user_text))
    if not value_words:
        return False
    for w in value_words:
        if w in user_words:
            continue
        if allow_word_endings and len(w) >= 5 and any(u[:5] == w[:5] for u in user_words):
            continue
        return False
    return True


def ground_candidate_details(args: dict, user_text: str) -> dict:
    """Keep only the extracted details that the user actually stated in this session."""
    grounded = {}
    if args.get("name") is not None and is_stated(args["name"], user_text):
        grounded["name"] = args["name"]
    if args.get("desired_role") is not None and is_stated(args["desired_role"], user_text, allow_word_endings=True):
        grounded["desired_role"] = args["desired_role"]
    years = args.get("experience_years")
    try:
        years = float(years)
        if years.is_integer() and int(years) in stated_years(user_text):
            grounded["experience_years"] = int(years)
    except (TypeError, ValueError):
        pass  # missing or not a number -> leave it out
    return grounded


def is_question_only(message: str) -> bool:
    """A question that does not describe the user (e.g. 'How can I improve my CV?')."""
    return "?" in message and not SELF_DESCRIPTION_PATTERN.search(message.lower())


def get_extraction_args(ai_message: AIMessage) -> Optional[dict]:
    """Arguments of an extract_candidate_info call, or candidate JSON the model wrote as text."""
    for call in ai_message.tool_calls:
        if call["name"] == "extract_candidate_info":
            return dict(call["args"])
    match = re.search(r"\{.*\}", to_text(ai_message), re.DOTALL)
    if match:
        try:
            data = json.loads(match.group())
        except json.JSONDecodeError:
            return None
        if isinstance(data, dict) and ({"name", "desired_role"} & data.keys()):
            return data
    return None


def to_text(ai_message: AIMessage) -> str:
    """Get plain text from a model reply (Gemini can return a list of content blocks)."""
    return (ai_message.text or "").strip()


def answer_without_tools(history: list[BaseMessage], message: str) -> str:
    """Plain-text answer used when a tool call was rejected."""
    reply = to_text(answer_chain.invoke({"history": history, "input": message}))
    return reply or "Could you tell me a bit more about what you'd like help with?"


def ask_for(fields: list[str]) -> str:
    labels = [FIELD_LABELS[f] for f in fields]
    return "Thanks! Could you tell me " + " and ".join(labels) + "?"


def chatbot_response(message: str, session_id: str = "default") -> str:
    """Send one user message to the chatbot and return either text or a JSON string."""
    message = message.strip()
    if not message:
        return "Please type a career question or tell me about the role you're looking for."
    history = get_memory(session_id)

    try:
        # 1-2. Send system prompt + history + new message to the model
        ai_message = chatbot_chain.invoke({"history": history, "input": message})
        args = get_extraction_args(ai_message)

        # Career question or ordinary message: answer in plain text
        if args is None or is_question_only(message):
            reply = to_text(ai_message) if args is None else ""
            reply = reply or answer_without_tools(history, message)
            save_to_memory(session_id, message, reply)
            return reply

        # 3. Keep only details the user actually stated, then merge with the saved profile
        user_text = " ".join([m.content for m in history if isinstance(m, HumanMessage)] + [message])
        profile = get_profile(session_id)
        changes = {k: v for k, v in ground_candidate_details(args, user_text).items() if profile.get(k) != v}
        if not changes:  # nothing new was stated -> not a profile update
            reply = answer_without_tools(history, message)
            save_to_memory(session_id, message, reply)
            return reply

        updated = {**profile, **changes}
        try:
            candidate = extract_candidate_info.invoke(updated)  # validates with CandidateInfo
        except ValidationError as err:
            # Keep the valid details, ask only for the missing or invalid ones
            bad_fields = [f for f in FIELD_LABELS if f in {e["loc"][0] for e in err.errors()}]
            profile_store[session_id] = {k: v for k, v in updated.items() if k not in bad_fields}
            reply = answer_without_tools(history, message) if "?" in message else ask_for(bad_fields)
            save_to_memory(session_id, message, reply)
            return reply
    except Exception as err:  # network / quota / auth problems
        return f"{ERROR_PREFIX} right now ({type(err).__name__}: {err})."

    # 4-5. Save the profile and return clean JSON
    profile_store[session_id] = candidate
    details = ", ".join(f"{key}={value}" for key, value in candidate.items())
    save_to_memory(session_id, message, f"(Candidate profile saved with extract_candidate_info: {details}.)")
    return json.dumps(candidate, indent=2)


def chat(message: str, session_id: str = "default") -> str:
    """Convenience wrapper: prints the exchange and returns the response."""
    response = chatbot_response(message, session_id)
    print(f"User: {message}\n\nAssistant:\n{response}\n")
    return response

## 9. Testing

Testing has two parts:

1. **Safeguard checks (9.1)** test the validation and grounding code directly, without calling the model.
2. **Live chatbot scenarios (9.2–9.6)** send real messages to Gemini. They cover the assignment requirement of three conversational questions and two structured extractions, plus memory, corrections, invented-detail checks and session isolation.

Each live scenario is verified automatically against several criteria, so a response cannot pass just because it is plain text:

| Test type | Pass criteria |
|---|---|
| Career question | Plain text, not JSON; no API error; mentions at least one topic keyword; does not ask for the user's name, role or experience; saved profile unchanged |
| Extraction | Valid JSON exactly matching the expected record (string values compared case-insensitively); `experience_years` is an integer or absent; saved profile matches |

Each candidate uses its own `session_id` to keep records isolated.

In [ ]:
test_results = []

# Requests aimed at the user (not advice such as "put your name at the top of your CV")
PERSONAL_DETAIL_REQUEST = re.compile(
    r"\bwhat(?:'s| is) your (?:full |target |desired |dream )?(?:name|role|job|position)\b"
    r"|\b(?:may|can|could) (?:i|you)\b[^.?!]{0,25}\byour (?:full )?(?:name|role|experience|background)\b"
    r"|\b(?:tell|let) me (?:know )?(?:about )?your (?:full )?(?:name|role|experience|background)\b"
    r"|\b(?:what|which) (?:role|job|position) are you\b"
    r"|\bhow many years[^.?!]*\byou have\b",
    re.IGNORECASE,
)


def is_json(text: str) -> bool:
    try:
        return isinstance(json.loads(text), dict)
    except (json.JSONDecodeError, TypeError):
        return False


def asks_for_personal_details(text: str) -> bool:
    return bool(PERSONAL_DETAIL_REQUEST.search(text))


def normalise(data: dict) -> dict:
    """Case-insensitive comparison for string values."""
    return {k: v.lower() if isinstance(v, str) else v for k, v in data.items()}


def record(title: str, checks: dict) -> None:
    for name, ok in checks.items():
        print(f"  [{'PASS' if ok else 'FAIL'}] {name}")
    passed = all(checks.values())
    print(f"RESULT: {'PASS' if passed else 'FAIL'}\n")
    test_results.append((title, passed))


def run_test(title, message, session_id, expected=None, keywords=(), allow_detail_request=False, profile_may_change=False):
    """Send one message and verify the response.

    expected             -> extraction test: dict the JSON output must match
    keywords             -> career test: the answer must mention at least one of these
    allow_detail_request -> the assistant may ask for missing details (only for partial profiles)
    """
    print("=" * 80)
    print(title)
    print("=" * 80)
    profile_before = dict(get_profile(session_id))
    response = chat(message, session_id)
    profile_after = get_profile(session_id)

    if expected is not None:
        data = json.loads(response) if is_json(response) else None
        checks = {
            "returns a JSON object": data is not None,
            f"JSON equals {expected}": data is not None and normalise(data) == normalise(expected),
            "experience_years is an integer or absent": data is None or type(data.get("experience_years", 0)) is int,
            "saved profile matches the JSON": normalise(profile_after) == normalise(expected),
        }
    else:
        checks = {
            "plain text, not JSON": not is_json(response),
            "no API error": not response.startswith(ERROR_PREFIX),
        }
        if keywords:
            checks[f"on topic (mentions one of {list(keywords)})"] = any(k in response.lower() for k in keywords)
        if not allow_detail_request:
            checks["does not ask for name, role or experience"] = not asks_for_personal_details(response)
        if not profile_may_change:
            checks[f"saved profile unchanged ({profile_before})"] = profile_after == profile_before
    record(title, checks)
    return response

### 9.1 Safeguard Checks (no API calls)

These checks run the validation and grounding code directly with fixed inputs, so they are deterministic and do not use the model.

In [ ]:
def check(title: str, condition: bool) -> None:
    print(f"[{'PASS' if condition else 'FAIL'}] {title}")
    test_results.append((title, condition))


def validation_fails(args: dict) -> bool:
    try:
        extract_candidate_info.invoke(args)
        return False
    except ValidationError:
        return True


# Experience durations
check("Safeguard: 'three years' is read as 3", stated_years("I have three years of experience") == {3})
check("Safeguard: '2 yrs' is read as 2", stated_years("about 2 yrs in retail") == {2})
check("Safeguard: 'no experience' is read as 0", stated_years("I have no professional experience yet") == {0})
check("Safeguard: 'What experience is needed?' states no duration", stated_years("What experience is needed?") == set())

# Grounding: only details the user stated survive
check(
    "Safeguard: example name is discarded when the user gave no name",
    ground_candidate_details({"name": "Sam", "desired_role": "Software Engineer"},
                             "What skills should I improve for a software engineering job?") == {"desired_role": "Software Engineer"},
)
check(
    "Safeguard: invented experience is discarded",
    "experience_years" not in ground_candidate_details({"name": "Ana", "desired_role": "Designer", "experience_years": 2},
                                                       "I'm Ana, applying as a designer. What experience is needed?"),
)
check(
    "Safeguard: stated details are kept (float 3.0 -> int 3)",
    ground_candidate_details({"name": "Sam", "desired_role": "Developer", "experience_years": 3.0},
                             "My name is Sam and I am applying for a developer role with three years of experience.")
    == {"name": "Sam", "desired_role": "Developer", "experience_years": 3},
)
check(
    "Safeguard: explicit zero years is kept",
    ground_candidate_details({"name": "Lee", "desired_role": "Tester", "experience_years": 0},
                             "I'm Lee, applying for a tester job with no experience.").get("experience_years") == 0,
)

# Personal-detail detector used by the live tests
check("Safeguard: detector flags 'What is your name?'", asks_for_personal_details("Great! What is your name and which role are you applying for?"))
check("Safeguard: detector ignores CV advice mentioning 'your name'", not asks_for_personal_details("Put your name and contact details at the top of your CV."))

# Question detection
check("Safeguard: career question is detected", is_question_only("How can I make my CV stronger?"))
check("Safeguard: self-description is not treated as a pure question", not is_question_only("I'm Tom, a nurse with 5 years. Any tips?"))

# Validation
check("Safeguard: blank name is rejected", validation_fails({"name": "   ", "desired_role": "Developer"}))
check("Safeguard: missing role is rejected", validation_fails({"name": "Sam"}))
check("Safeguard: non-numeric experience is rejected", validation_fails({"name": "Sam", "desired_role": "Dev", "experience_years": "lots"}))
check("Safeguard: whitespace is stripped", extract_candidate_info.invoke({"name": "  Sam ", "desired_role": " developer "}) == {"name": "Sam", "desired_role": "Developer"})

### 9.2 Career Questions (Tests 1–3)

Three career questions in a fresh session. Each answer must be relevant advice that does not ask for personal details.

In [ ]:
clear_memory("career-qa")
run_test("TEST 1 – Career question: interview preparation", "How do I prepare for a data analyst interview?",
         session_id="career-qa", keywords=("interview", "sql", "star", "practi", "project"))
run_test("TEST 2 – Career question: software engineering skills", "What skills should I improve for a software engineering job?",
         session_id="career-qa", keywords=("skill", "coding", "algorithm", "data structure", "system design", "git"))
run_test("TEST 3 – Career question: CV improvement", "How can I make my CV stronger?",
         session_id="career-qa", keywords=("cv", "resume", "achievement", "tailor", "quantif"));

### 9.3 Structured Extraction (Tests 4–5)

Test 4 states the experience in words ("three years"), which must become the integer `3`. Test 5 states no experience, so `experience_years` must be omitted rather than set to `0`.

In [ ]:
clear_memory("candidate-sam")
run_test(
    "TEST 4 – Extraction: complete profile, experience in words",
    "My name is Sam and I am applying for a developer role with three years of experience.",
    session_id="candidate-sam",
    expected={"name": "Sam", "desired_role": "Developer", "experience_years": 3},
)

clear_memory("candidate-sarah")
run_test(
    "TEST 5 – Extraction: no experience stated -> field omitted",
    "My name is Sarah and I am applying for a marketing analyst role.",
    session_id="candidate-sarah",
    expected={"name": "Sarah", "desired_role": "Marketing Analyst"},
);

### 9.4 Memory and Corrections (Test 6)

The candidate provides details across three messages:

| Step | Message | Expected behaviour |
|---|---|---|
| 6a | Name only | The assistant asks for the desired role; no JSON is produced |
| 6b | Role and experience | Both turns are combined into a single record |
| 6c | Correction from three to four years | Only `experience_years` changes; name and role are retained |

In [ ]:
clear_memory("candidate-john")

run_test("TEST 6a – Memory: name only (no JSON yet)", "My name is John.",
         session_id="candidate-john", keywords=("role", "position", "job"),
         allow_detail_request=True, profile_may_change=True)

run_test("TEST 6b – Memory: combine with the previous turn",
         "I am applying for a business analyst role and I have three years of experience.",
         session_id="candidate-john",
         expected={"name": "John", "desired_role": "Business Analyst", "experience_years": 3})

run_test("TEST 6c – Memory: correction from three to four years",
         "Sorry, correction: I have four years of experience, not three.",
         session_id="candidate-john",
         expected={"name": "John", "desired_role": "Business Analyst", "experience_years": 4})

### 9.5 Career Questions After and Without a Profile (Tests 7–9)

| Test | Situation | Expected behaviour |
|---|---|---|
| 7 | Career question after John's profile was saved | Advice, not profile JSON; profile unchanged |
| 8 | Career question in a new session, no name given | Advice; no profile created; no example names in the reply |
| 9 | "What experience is needed?" after Sarah's profile (no experience stated) | Advice; no `experience_years` value invented |

In [ ]:
run_test("TEST 7 – Career question after extraction", "How should I prepare for a business analyst interview?",
         session_id="candidate-john", keywords=("interview", "stakeholder", "requirement", "case", "star", "practi", "sql"))

clear_memory("no-name")
response = run_test("TEST 8 – Career question with no name supplied", "What should I include in a cover letter?",
                    session_id="no-name", keywords=("cover letter", "company", "skill", "achievement", "role"))
check("TEST 8 – no profile was created", get_profile("no-name") == {})
check("TEST 8 – no names from the examples or other sessions in the reply", not re.search(r"\b(Maya|Priya|Sam)\b", response))

run_test("TEST 9 – 'What experience is needed?' without a stated duration", "What experience is needed?",
         session_id="candidate-sarah", keywords=("skill", "project", "portfolio", "internship", "entry", "tool", "campaign", "analyt", "degree"))
check("TEST 9 – Sarah's profile still has no experience_years", "experience_years" not in get_profile("candidate-sarah"));

### 9.6 Session Isolation and Memory Clearing (Test 10)

Each candidate's profile must be independent. After `clear_memory`, the session must have no messages and no profile, and the assistant must not recall the earlier candidate.

In [ ]:
print("Profiles by session:")
for sid in ("candidate-sam", "candidate-sarah", "candidate-john"):
    print(f"  {sid:<16} {get_profile(sid)}")
print()

check("TEST 10 – sessions hold separate profiles",
                get_profile("candidate-sam").get("name") == "Sam"
                and get_profile("candidate-sarah").get("name") == "Sarah"
                and get_profile("candidate-john").get("name") == "John")

clear_memory("candidate-john")
check("TEST 10 – clear_memory empties messages and profile",
                "candidate-john" not in memory_store and "candidate-john" not in profile_store)

response = chat("Do you remember my name?", session_id="candidate-john")
check("TEST 10 – after clearing, the earlier name is not recalled", "john" not in response.lower())
check("TEST 10 – after clearing, no profile is created", get_profile("candidate-john") == {})
check("TEST 10 – other sessions are unaffected", get_profile("candidate-sam").get("name") == "Sam")

### 9.7 Test Summary

In [ ]:
print(f"{'Test':<75} Result")
print("-" * 85)
for title, passed in test_results:
    print(f"{title:<75} {'PASS' if passed else 'FAIL'}")
print("-" * 85)
print(f"{sum(p for _, p in test_results)}/{len(test_results)} checks passed")

## 10. Gradio Interface (Optional)

A chat interface built on the same `chatbot_response` function. Candidate extractions are displayed as formatted JSON blocks.

- Each browser session maintains its own memory, keyed by Gradio's `session_hash`.
- Clearing the chat also clears the associated memory.
- This section is independent of the core implementation and can be skipped.

In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr


def gradio_chat(message: str, history: list, request: gr.Request) -> str:
    session_id = f"gradio-{request.session_hash}" if request else "gradio"
    if not history:  # new or cleared chat -> start with empty memory
        clear_memory(session_id)

    response = chatbot_response(message, session_id)
    # Show extractions as a JSON code block, normal answers as-is
    return f"```json\n{response}\n```" if is_json(response) else response


demo = gr.ChatInterface(
    fn=gradio_chat,
    title="Recruitment Assistant",
    description="Ask career questions, or tell me your name, the role you want and your experience to get a structured JSON profile.",
    examples=[
        "How do I prepare for a data analyst interview?",
        "How can I make my CV stronger?",
        "My name is Sam and I am applying for a developer role with 3 years of experience.",
    ],
)

demo.launch()

## 11. Reflection

**Function calling and response consistency.** Defining `extract_candidate_info` as a tool, with the Pydantic `CandidateInfo` model as its schema, made every extraction follow the same structure. Types and required fields are enforced, blank values are rejected, roles are formatted consistently, and `experience_years` is simply omitted when unknown. Function calling alone did not guarantee correct behaviour, though. My first prompt told the model to call the tool as soon as it knew a name and role. In an early run this made the CV and software-engineering questions return a JSON profile for "Sam", a name that came from my own few-shot example rather than from the user. The fix combined a clearer trigger rule (only when the current message supplies or corrects details) with code that accepts a detail only if the user actually stated it, including an explicit duration for experience.

**Prompt engineering, tone and formatting.** The role prompt and few-shot examples produced a consistently warm tone and a predictable layout of a short opening line followed by bullet points. The same run also showed how strongly examples shape behaviour: an example name was copied into real output. I changed the examples to fictional names that are not used in testing and added two examples in which a career question gets advice rather than a tool call.

**Production improvement.** Memory and profiles are currently stored in Python dictionaries and are lost when the runtime restarts. A production version would store conversations and candidate records in a database, send validated records to the agency's applicant tracking system, and ask candidates to confirm their details before submission.